In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import os, warnings
warnings.filterwarnings('ignore')

POSSIBLE_PATHS = [
    '/kaggle/input/datasets/quangleuq/v3-data/',
    '/kaggle/input/gesturhythm-v2/',
    '/kaggle/input/v3-data/'
]

DATA_DIR = next((p for p in POSSIBLE_PATHS if os.path.isdir(p)), '.')
print('Using data dir:', DATA_DIR)

sequences = np.load(os.path.join(DATA_DIR, 'sequences.npy'))   # (N,30,225)
emotions  = np.load(os.path.join(DATA_DIR, 'emotions.npy'))    # (N,2)
mode_ids  = np.load(os.path.join(DATA_DIR, 'mode_ids.npy'))    # (N,)

N = len(sequences)
valence, arousal = emotions[:, 0], emotions[:, 1]

MODE_NAMES  = {0:'HAPPY_HIGH',1:'HAPPY_LOW',2:'SAD_HIGH',3:'SAD_LOW',4:'NEUTRAL',5:'NONE'}
MODE_COLORS = {0:'#2ecc71',1:'#3498db',2:'#e74c3c',3:'#9b59b6',4:'#f39c12',5:'#95a5a6'}
print(f'Loaded: sequences{sequences.shape}, emotions{emotions.shape}, modes{mode_ids.shape}')


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Scatter valence vs arousal
ax = axes[0]
for mid, mname in MODE_NAMES.items():
    mask = mode_ids == mid
    if mask.sum() == 0: continue
    ax.scatter(valence[mask], arousal[mask], c=MODE_COLORS[mid], label=mname, alpha=0.6, s=20)
ax.set_title('Phân bố Valence vs Arousal theo Mode')
ax.set_xlabel('Valence')
ax.set_ylabel('Arousal')
ax.legend(fontsize=7)

# Histogram valence
ax = axes[1]
for mid, mname in MODE_NAMES.items():
    mask = mode_ids == mid
    if mask.sum() == 0: continue
    ax.hist(valence[mask], bins=20, color=MODE_COLORS[mid], label=mname, alpha=0.5)
ax.set_title('Phân bố Valence theo Mode')
ax.set_xlabel('Valence')
ax.set_ylabel('Số lượng')
ax.legend(fontsize=7)

# Histogram arousal
ax = axes[2]
for mid, mname in MODE_NAMES.items():
    mask = mode_ids == mid
    if mask.sum() == 0: continue
    ax.hist(arousal[mask], bins=20, color=MODE_COLORS[mid], label=mname, alpha=0.5)
ax.set_title('Phân bố Arousal theo Mode')
ax.set_xlabel('Arousal')
ax.set_ylabel('Số lượng')
ax.legend(fontsize=7)

plt.tight_layout()
plt.savefig('emotion_distribution.png', dpi=150)
plt.show()
print('Saved: emotion_distribution.png')


In [ ]:
np.random.seed(42)
idxs = np.random.choice(N, min(5, N), replace=False)
fig, axes = plt.subplots(5, 1, figsize=(14, 12), sharex=True)

for i, idx in enumerate(idxs):
    seq = sequences[idx]          # (30, 225)
    mid = int(mode_ids[idx])
    # plot mean across all 225 landmarks per frame
    mean_per_frame = seq.mean(axis=1)
    axes[i].plot(mean_per_frame, color=MODE_COLORS[mid], linewidth=1.5)
    axes[i].set_ylabel('Trung bình\nLandmark', fontsize=8)
    axes[i].set_title(f'Sequence #{idx} — Mode: {MODE_NAMES[mid]}  |  V={emotions[idx,0]:.2f}, A={emotions[idx,1]:.2f}', fontsize=9)
    axes[i].grid(True, alpha=0.3)

axes[-1].set_xlabel('Frame')
fig.suptitle('Ví dụ 5 Gesture Sequence Khác Nhau', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('gesture_sequences.png', dpi=150)
plt.show()
print('Saved: gesture_sequences.png')


In [ ]:
# Sample landmarks to keep heatmap readable (take every 15th -> 15 landmarks)
step = 15
lm_indices = list(range(0, 225, step))  # 15 landmarks
flat = sequences[:, :, lm_indices].reshape(N, -1)  # (N, 30*15)
# Correlation across sequences using mean per landmark group
lm_mean = sequences[:, :, lm_indices].mean(axis=1)  # (N, 15)
corr = np.corrcoef(lm_mean.T)                        # (15, 15)

labels = [f'LM{i}' for i in lm_indices]
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, xticklabels=labels, yticklabels=labels,
            cmap='coolwarm', center=0, vmin=-1, vmax=1,
            annot=True, fmt='.1f', annot_kws={'size': 7}, ax=ax)
ax.set_title('Heatmap Tương Quan Giữa Các Landmarks (mẫu mỗi 15)', fontsize=12)
ax.set_xlabel('Landmark Index')
ax.set_ylabel('Landmark Index')
plt.tight_layout()
plt.savefig('landmark_heatmap.png', dpi=150)
plt.show()
print('Saved: landmark_heatmap.png')


In [ ]:
# Mean of first 9 landmark coords per mode
n_lm = 9
present_modes = [m for m in range(6) if (mode_ids == m).sum() > 0]
mode_means = []
mode_labels = []
for mid in present_modes:
    mask = mode_ids == mid
    mean_vals = sequences[mask].mean(axis=(0, 1))[:n_lm]  # (9,)
    mode_means.append(mean_vals)
    mode_labels.append(MODE_NAMES[mid])

mode_means = np.array(mode_means)  # (M, 9)
x = np.arange(n_lm)
width = 0.8 / len(present_modes)

fig, ax = plt.subplots(figsize=(13, 5))
for i, (mid, label) in enumerate(zip(present_modes, mode_labels)):
    offset = (i - len(present_modes)/2 + 0.5) * width
    ax.bar(x + offset, mode_means[i], width, label=label, color=MODE_COLORS[mid], alpha=0.8)

ax.set_title('So Sánh Trung Bình Landmarks Giữa 6 Mode Cảm Xúc (9 Landmarks Đầu)', fontsize=11)
ax.set_xlabel('Landmark Index')
ax.set_ylabel('Giá trị trung bình')
ax.set_xticks(x)
ax.set_xticklabels([f'LM{i}' for i in range(n_lm)])
ax.legend()
plt.tight_layout()
plt.savefig('mode_comparison.png', dpi=150)
plt.show()
print('Saved: mode_comparison.png')


In [ ]:
# No person_ids in dataset — split by index as proxy for Person A / Person C
mid_point = N // 2
person_a = sequences[:mid_point]
person_c = sequences[mid_point:]

mean_a = person_a.mean(axis=(0, 1))[:15]
mean_c = person_c.mean(axis=(0, 1))[:15]
x = np.arange(15)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(x - 0.2, mean_a, 0.4, label='Người A (nửa đầu)', color='#3498db', alpha=0.8)
axes[0].bar(x + 0.2, mean_c, 0.4, label='Người C (nửa sau)', color='#e74c3c', alpha=0.8)
axes[0].set_title('So Sánh Trung Bình Landmarks: Người A vs Người C')
axes[0].set_xlabel('Landmark Index')
axes[0].set_ylabel('Giá trị trung bình')
axes[0].set_xticks(x)
axes[0].set_xticklabels([f'LM{i}' for i in range(15)], rotation=45)
axes[0].legend()

# Valence/arousal distribution
em_a = emotions[:mid_point]
em_c = emotions[mid_point:]
axes[1].scatter(em_a[:, 0], em_a[:, 1], alpha=0.5, s=15, label='Người A', color='#3498db')
axes[1].scatter(em_c[:, 0], em_c[:, 1], alpha=0.5, s=15, label='Người C', color='#e74c3c')
axes[1].set_title('Phân bố Cảm Xúc: Người A vs Người C')
axes[1].set_xlabel('Valence')
axes[1].set_ylabel('Arousal')
axes[1].legend()

plt.tight_layout()
plt.savefig('person_comparison.png', dpi=150)
plt.show()
print('Saved: person_comparison.png')
print('Lưu ý: Không có person_ids — chia đôi dataset làm proxy Người A / Người C')


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Missing values per feature
flat_seq = sequences.reshape(N, -1)
missing_per_feat = np.isnan(flat_seq).sum(axis=0)
axes[0].plot(missing_per_feat, color='#e74c3c', linewidth=0.8)
axes[0].set_title('Missing Values Theo Từng Feature')
axes[0].set_xlabel('Feature Index')
axes[0].set_ylabel('Số lượng NaN')
axes[0].text(0.5, 0.95, f'Tổng NaN: {missing_per_feat.sum()}',
             transform=axes[0].transAxes, ha='center', va='top', fontsize=10)

# Outliers: values beyond 3 std
mu, sigma = flat_seq.mean(), flat_seq.std()
outlier_counts = ((np.abs(flat_seq - mu) > 3 * sigma).sum(axis=1))
axes[1].hist(outlier_counts, bins=30, color='#f39c12', edgecolor='white')
axes[1].set_title('Phân bố Số Outlier (>3σ) Theo Sequence')
axes[1].set_xlabel('Số outlier per sequence')
axes[1].set_ylabel('Số lượng sequence')

# Frame coverage: non-zero frames per sequence
frame_nonzero = (sequences.reshape(N, 30, -1).sum(axis=2) != 0).sum(axis=1)
axes[2].hist(frame_nonzero, bins=30, color='#2ecc71', edgecolor='white')
axes[2].set_title('Phân bố Frame Coverage (Frames Khác 0) Mỗi Sequence')
axes[2].set_xlabel('Số frame có dữ liệu')
axes[2].set_ylabel('Số lượng sequence')

plt.tight_layout()
plt.savefig('data_quality.png', dpi=150)
plt.show()
print('Saved: data_quality.png')


In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
for mid, mname in MODE_NAMES.items():
    mask = mode_ids == mid
    if mask.sum() == 0: continue
    ax.scatter(valence[mask], arousal[mask],
               c=MODE_COLORS[mid], label=f'{mname} (n={mask.sum()})',
               alpha=0.6, s=25, edgecolors='none')

ax.axhline(0, color='gray', linewidth=0.8, linestyle='--')
ax.axvline(0, color='gray', linewidth=0.8, linestyle='--')
ax.set_title('Toàn Bộ Dữ Liệu Trong Không Gian Valence–Arousal (Màu theo Mode)', fontsize=12)
ax.set_xlabel('Valence (Tích cực ↔ Tiêu cực)')
ax.set_ylabel('Arousal (Kích thích ↔ Bình lặng)')
ax.legend()
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig('emotion_space_2d.png', dpi=150)
plt.show()
print('Saved: emotion_space_2d.png')


In [ ]:
# Sequence 'effective length' = number of frames where any landmark is non-zero
eff_len = (sequences.reshape(N, 30, -1).abs() if hasattr(sequences, 'abs')
           else np.abs(sequences.reshape(N, 30, -1)))
eff_len = (eff_len.sum(axis=2) > 0).sum(axis=1)  # (N,)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].hist(eff_len, bins=30, color='#3498db', edgecolor='white')
axes[0].set_title('Phân bố Độ Dài Sequence (Số Frame Có Dữ Liệu)')
axes[0].set_xlabel('Độ dài sequence (frames)')
axes[0].set_ylabel('Số lượng sequence')
axes[0].axvline(eff_len.mean(), color='red', linestyle='--', label=f'Mean={eff_len.mean():.1f}')
axes[0].legend()

# By mode
for mid, mname in MODE_NAMES.items():
    mask = mode_ids == mid
    if mask.sum() == 0: continue
    axes[1].hist(eff_len[mask], bins=20, color=MODE_COLORS[mid], label=mname, alpha=0.5)
axes[1].set_title('Phân bố Độ Dài Sequence Theo Mode')
axes[1].set_xlabel('Độ dài sequence (frames)')
axes[1].set_ylabel('Số lượng sequence')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig('sequence_length_dist.png', dpi=150)
plt.show()
print('Saved: sequence_length_dist.png')
